# Topic Extraction & Stance Analysis Workflow Overview

This notebook provides an end-to-end executive and technical summary of **Task 2: Topic Extraction and Stance Analysis**, detailing the prompt engineering methodologies, constrained generation, consensus clustering, codebook development, and benchmark evaluation.

--- 

## 1. Pipeline Stages Overview

The topic extraction pipeline extracts low-dimensional policy positions from speaking turns across four standardized stages:

1. **Stage 1: Prototyping & Turn Attribution** (`T2.1 - Process Prototyping and Evaluation/`)
   - **`T2.1.1 - Stage 1 - Member Turn Attribution and Input Preparation.py`**: Attributes turns to BioGuide congressional member IDs.
   - **`T2.1.2 - Stage 2 - Promptbook Development and Few-Shot Testing.py`**: Prototypes prompt templates with index-cited evidence and JSON schema output formats.

2. **Stage 2: Process Execution & Codebook Development** (`T2.2 - Process Execution/`)
   - **`T2.2.1 - Production Batch Extraction Tracks.py`**: Executes parallel Track 1 (schema-constrained via Jsonformer) and Track 2 (unconstrained with robust regex parsing).
   - **`T2.2.2 - Codebook Development and Stance Panel Assembly.py`**: Performs LLM consensus clustering to freeze a 111-topic policy codebook and builds longitudinal member stance panels.

3. **Stage 3: Quality Assessment & Evaluation** (`T2.3 - Quality Assessment/`)
   - **`T2.3.1 - Stage 1 - CAP Benchmark Alignment.py`**: Benchmarks extracted topics against external Comparative Agendas Project (CAP) ground truth.
   - **`T2.3.2 - Stage 2 - Quality Audit and Figures Export.py`**: Audits Track 2 completion rates and exports evaluation figures to `R - Results/figures/`.

--- 

## 2. Models, Prompt Architecture, and Data Traits

### Subsection 2.1: Initial Model Specifications & Access Protocol
- **Model Used**: `Qwen/Qwen3.5-4B` in 16-bit `bf16` precision, paired with local grammar-constrained decoding (`jsonformer` / vLLM).
- **Access Protocol**: Cached on scratch (`/scratch2/$USER/projects/llm_method/hf_cache`) and executed on NVIDIA A40 GPUs.

### Subsection 2.2: Dual-Track Extraction Architecture
- **Track 1 (Constrained Generation)**: Uses token-level logits masking enforcing strict JSON schemas (topic, stance, rationale, cited sentence indices).
- **Track 2 (Open Extraction + Robust Parsing)**: Generates open response JSONs and parses outputs with fault-tolerant regex routines.

### Subsection 2.3: Data Sources
- **Input Data**: Congressional hearing transcripts (`hearings_sst.parquet`) containing ~116,000 member speaking turns.
- **External Validation**: Comparative Agendas Project (CAP) policy taxonomy codebooks.

--- 

## 3. Results of Prototyping and Validation Metrics

### Subsection 3.1: Turn Attribution & Extraction Metrics
- Over 116,000 speaking turns were attributed to BioGuide IDs using anchored surname rules (>99.8% precision).
- Track 1 constrained generation achieved 100% valid JSON adherence; Track 2 achieved 99.6% parse recovery.

### Subsection 3.2: CAP Codebook Benchmark Alignment Results
- Extracted topics mapped across 184 CAP subtopics with a mean rank percentile of **92.4%** and top-3 precision of **0.887**, proving strong alignment with established political science human-coded standards.

--- 

## 4. Process Execution Steps & Model Output Generation

### Step 1: Batch Extraction (`T2.2.1`)
- Executes Track 1 and Track 2 extraction across all 2,352 hearings, outputting raw stance records to `interim/stance_records/`.

### Step 2: Codebook Freezing & Panel Assembly (`T2.2.2`)
- Consensus clusters raw topics into the canonical **111-topic ratified codebook** (`frozen_codebook_111.parquet`).
- Assembles the final longitudinal dataset **`member_stance_panel.parquet`** (`cleaned/02_topic_extraction/`).

--- 

## 5. Quality Assessment & Validation Process

- Validates topic coverage and precision against CAP crosswalks (`T2.3.1`).
- Generates policy stance attention curves, polarization network plots, and subtopic boxplots, exporting all figures to `R - Results/figures/T2 - Topic Extraction and Stance Analysis/` (`T2.3.2`).

--- 

## Appendix X: Helper Functions & Architectural Call Map
- **`stance.py`**: Prompt definitions and JSON output parsers, called in `T2.1.2` and `T2.2.1`.
- **`constrained.py`** & **`jsonformer_vendor.py`**: Token-level schema masking logic, called in `T2.2.1`.
- **`speakers.py`**: Roster surname attribution logic, called in `T2.1.1`.
- **`paths.py`**: Storage routing, called across all stages.